# Deep Ensembles: train members 5-14 on GPU (CISC 473 P23)

Trains ResNet-20 members with seeds 5-9 (ensemble B + the M=10 ablation) then 10-14 (ensemble C), exactly as the
midterm recipe (CIFAR-10 60/20/20 split seed 0, SGD-Nesterov lr 0.1, wd 5e-4, bs 128, cosine 40 epochs, crop+flip).
Then dumps softmax probabilities and zips everything for download.

**How to run (Kaggle or Colab)**
1. Push branch `de-final` to GitHub first (the notebook clones it) and set `REPO_URL` / `BRANCH` below.
2. Enable a GPU (Kaggle: Settings > Accelerator > GPU T4/P100; Colab: Runtime > Change runtime type) and **Internet ON**
   (needed for `git clone`, the torch.hub model definition, and the SVHN test download).
3. Run all cells. Training is resumable: if the session dies, re-run the same cells (finished seeds are skipped, partial ones resume).
   Expect roughly 10-25 min per member on a T4 (data augmentation runs on the CPU, so a GPU does not make it 100x faster); 10 members, a few hours.
4. Download `de_gpu_outputs.zip` (written after seeds 5-9 and again after 10-14) and unzip it **at the repo root** on your laptop:
   `unzip de_gpu_outputs.zip` (it contains `results/raw/de_members/seed{5..14}/` and `models/deep_ensembles/resnet20_seed*.pt`, train logs and times).
5. On the laptop run `python src/de_eval.py`: ensembles B, C and the M>5 ablation are picked up automatically.
Note: GPU training is not bit-identical to the CPU runs for seeds 0-4 (different kernels); only the seed, split and recipe are shared.

In [ ]:
REPO_URL = "https://github.com/spy772/CISC-473-P23.git"
BRANCH = "de-final"
GROUP1 = [5, 6, 7, 8, 9]       # ensemble B (+ M=10 ablation with seeds 0-4)
GROUP2 = [10, 11, 12, 13, 14]  # ensemble C

In [ ]:
import os, subprocess, sys
if not os.path.exists("CISC-473-P23"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "CISC-473-P23"], check=True)
os.chdir("CISC-473-P23")
# Kaggle/Colab already ship torch/numpy; only add what the dump script needs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "scikit-learn", "pandas", "matplotlib"], check=True)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Enable a GPU accelerator"
assert os.path.exists("data/cifar-10-batches-py/data_batch_1"), "CIFAR-10 batches are tracked in the repo"

In [ ]:
# SVHN official test split (26,032 images) is downloaded by the dump script; fetch it now to fail early.
subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0,'src'); import dump_member_probs as d; print(d.load_svhn_test().shape)"], check=True)

In [ ]:
def train_and_dump(seeds):
    s = [str(x) for x in seeds]
    subprocess.run([sys.executable, "models/deep_ensembles/train_ensemble.py", "--seeds"] + s, check=True)
    subprocess.run([sys.executable, "src/dump_member_probs.py", "--seeds"] + s, check=True)

def make_zip():
    import glob, re, zipfile
    files = (glob.glob("results/raw/de_members/seed*/*") + glob.glob("models/deep_ensembles/resnet20_seed*.pt")
             + glob.glob("models/deep_ensembles/train_log_seed*.csv") + glob.glob("models/deep_ensembles/train_time_seed*.json"))
    keep = [f for f in files if int(re.search(r"seed(\d+)", f.replace("\\", "/")).group(1)) >= 5 and not f.endswith(".partial.pt")]  # seeds 0-4 already on the laptop
    with zipfile.ZipFile("de_gpu_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in keep:
            z.write(f)
    print("wrote de_gpu_outputs.zip:", len(keep), "files,", os.path.getsize("de_gpu_outputs.zip") / 1e6, "MB")


In [ ]:
# --- Group 1: seeds 5-9 (ensemble B) ---
train_and_dump(GROUP1)
make_zip()   # download this now if you are short on time

In [ ]:
# --- Group 2: seeds 10-14 (ensemble C) ---
train_and_dump(GROUP2)
make_zip()

In [ ]:
import json, glob
print("seed  train_min   test_acc   hardware")
for s in GROUP1 + GROUP2:
    t = json.load(open("models/deep_ensembles/train_time_seed%d.json" % s))
    m = json.load(open("results/raw/de_members/seed%d/meta.json" % s))
    print("%4d  %9.1f   %.4f     %s" % (s, t["wall_clock_sec"] / 60, m["test_acc"], m["hardware"]))
# Colab: from google.colab import files; files.download("de_gpu_outputs.zip")   (Kaggle: file appears under Output)